In [9]:
import numpy as np

# Data

In [10]:
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

In [11]:
y = np.array([[0],
              [1],
              [1],
              [0]])

# Exercise

For the supplied XOR dataset, implement a simple feedforward neural network with one hidden layer with 2 units using sigmoid activation and 1 output unit with sigmoid activation. Train it using the MSE loss function and stochastic gradient descent (SGD), initializing the weights to small random values. Use a learning rate of 0.1, batch size of 2, and train for 10 epochs. After training, report the final weights and the predicted outputs for the XOR inputs.

## Solution

In [18]:
# Sigmoid activation and its derivative
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_deriv(z):
    a = sigmoid(z)
    return a * (1 - a)

# Reproducibility
np.random.seed(42)

# Network dimensions
input_size  = 2
hidden_size = 2
output_size = 1

# Weight initialisation (small random values)
W1 = np.random.randn(hidden_size, input_size) * 0.1   # (2, 2)
b1 = np.zeros((hidden_size, 1))                        # (2, 1)
W2 = np.random.randn(hidden_size, output_size) * 0.1  # (2, 1)
b2 = np.zeros((output_size, 1))                        # (1, 1)

# Hyper-parameters
lr         = 0.1
batch_size = 2
epochs     = 10000
n_samples  = X.shape[0]

W1, b1, W2, b2

(array([[ 0.04967142, -0.01382643],
        [ 0.06476885,  0.15230299]]),
 array([[0.],
        [0.]]),
 array([[-0.02341534],
        [-0.0234137 ]]),
 array([[0.]]))

Layers
$$
\mathbf{h}^{(1)} = \sigma \left( \mathbf{W}^{(1)} \mathbf{x} + \mathbf{b}^{(1)} \right)
$$
$$
\mathbf{h}^{(2)} = \sigma \left( \mathbf{W}^{(2)} \mathbf{h}^{(1)} + \mathbf{b}^{(2)} \right)
$$

In [ ]:
# Hidden layer
def z1(x, W1, b1):
    return W1 @ x + b1
def h1(x, W1, b1):
    return sigmoid(z1(x, W1, b1))

# Output layer
def z2(x, W2, b2):
    return W2 @ x + b2
def h2(x, W2, b2):
    return sigmoid(z2(x.reshape(2, 1), W2, b2))

Model
$$
\mathbf x \rightarrow \mathbf{z}^{(1)} \rightarrow \mathbf{h}^{(1)} \rightarrow \mathbf{z}^{(2)} \rightarrow \mathbf{h}^{(2)}
$$

In [20]:
# Model
def forward(x, W1, b1, W2, b2):
    z_1 = z1(x, W1, b1)
    h_1 = h1(x, W1, b1)
    z_2 = z2(h_1, W2, b2)
    h_2 = h2(h_1, W2, b2)
    return z_1, h_1, z_2, h_2

Loss
$$
L = \frac{1}{2} \left\| \mathbf{h}^{(2)} - \mathbf{y} \right\|^2
$$
$$
\frac{\partial L}{\partial \mathbf{h}^{(2)}} = \mathbf{h}^{(2)} - \mathbf{y}
$$

In [ ]:
def loss(y, h2):
    return 0.5 * np.sum((h2 - y) ** 2)
def loss_deriv(y, h2):
    return np.array(h2 - y).reshape(1, 1)

(1, 1)

Propagation errors
$$
\boldsymbol\delta^{(2)} = \frac{\partial L}{\partial \mathbf{z}^{(2)}} = \frac{\partial L}{\partial \mathbf{h}^{(2)}} \text{diag} \left( \sigma'(\mathbf{z}^{(2)}) \right) = \left( \mathbf{h}^{(2)} - \mathbf{y} \right)^\top \odot \text{diag} \left( \sigma'(\mathbf{z}^{(2)}) \right) = (y - h_2)\sigma'(z_2)
$$
$$
\boldsymbol\delta^{(1)} = \frac{\partial L}{\partial \mathbf{z}^{(1)}} = \boldsymbol\delta^{(2)} \left( \mathbf{W}^{(2)} \right)^\top \left( \sigma'(\mathbf{z}^{(1)}) \right)^\top = \left( \boldsymbol\delta^{(2)} \left( \mathbf{W}^{(2)} \right)^\top \right) \odot \left( \sigma'(\mathbf{z}^{(1)}) \right)^\top
$$

In [ ]:
def delta_2(y, h2, z2):
    return (y - h2) * (sigmoid_deriv(z2)).T

def delta_1(delta_2, W2, z1):
    return (delta_2 @ W2.T) * sigmoid_deriv(z1).T

array([[0., 0.]])

Gradients

In [ ]:
# W_1
def d_L_W1(delta_1, x):
    return delta_1.T @ x.T
# b_1
def d_L_b1(delta_1):
    return delta_1.T

# W_2
def d_L_W2(delta_2, h1):
    return delta_2.T @ h1.T
# b_2
def d_L_b2(delta_2):
    return delta_2.T